# COMP2701 Assessment 1: Build and Evaluate a Local RAG System for OrbitOps

**Student Starter Notebook**

In this assessment task, you will build a Retrieval-Augmented Generation (RAG) pipeline for a fictional company called **OrbitOps**.

You must use **open-source models only**. Do **not** use any hosted API.



### Data model requirement

All chunks and retrieval outputs must use the following structure:

Document:
- text: str
- metadata: dict
    - doc_id
    - title
    - chunk_id
    - source_file

All retrieval functions must return:
list[Document]

Mixing dictionaries, strings, and Document objects will result in errors and may lose marks.

## 0. Setup
Use only open-source tools. No API key is required.

In [12]:
# Uncomment if needed in a fresh environment
!pip install sentence-transformers rank-bm25 scikit-learn numpy pandas chromadb

In [13]:
import os
import re
import json
import uuid
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
DATA_DIR = Path("data/docs")
PUBLIC_EVAL_PATH = Path("data/public_eval.json")
CHROMA_DIR = Path("chroma_store")
COLLECTION_NAME = "orbitops_assignment"

with open(PUBLIC_EVAL_PATH, "r", encoding="utf-8") as f:
    public_eval = json.load(f)

print("Docs:", len(list(DATA_DIR.glob("*.txt"))))

Docs: 30


## 1. Data loading and Parsing

Each document has lightweight metadata at the top.

Example:
- Title
- Doc ID
- Date
- Team
- Tags

### Task
Implement `load_documents()` so that it returns a list of dictionaries with at least:
- `doc_id`
- `title`
- `date`
- `team`
- `tags`
- `text`
- `source_file`


In [15]:
@dataclass
class Document:
    text: str
    metadata: Dict[str, Any]

In [16]:
def parse_metadata_and_text(raw_text: str) -> dict:
    """Parse the metadata header and body from a document string.

    Returns a dictionary with keys:
    doc_id, title, date, team, tags, text
    """
    parsed_dict = {}
    for line in raw_text.splitlines():
      if line.startswith("Title:"):
        parsed_dict["title"] = line.split(": ", 1)[1]
      elif line.startswith("Doc ID:"):
        parsed_dict["doc_id"] = line.split(": ", 1)[1]
      elif line.startswith("Date:"):
        parsed_dict["date"] = line.split(": ", 1)[1]
      elif line.startswith("Team:"):
        parsed_dict["team"] = line.split(": ", 1)[1]
      elif line.startswith("Tags:"):
        parsed_dict["tags"] = line.split(": ", 1)[1]
      elif "text" in parsed_dict:
        parsed_dict["text"] += " " + line
      else:
        parsed_dict["text"] = line
    return parsed_dict

In [17]:
def load_documents(data_dir: Path) -> list[dict]:
    """ Load raw documents from data_dir as plain dicts. Convert to Document objects during chunking."""
    doc_list = []
    for filename in data_dir.glob("*.txt"):
        text = filename.read_text(encoding="utf-8")
        parsed_dict = parse_metadata_and_text(text)
        doc_list.append(parsed_dict)
    return doc_list

In [18]:
# Inspect two documents
docs = load_documents(DATA_DIR)
print(f"Loaded {len(docs)} documents")
docs[:2]

Loaded 30 documents


[{'title': 'Security Review Transcript',
  'doc_id': 'doc10',
  'date': '2026-02-05',
  'team': 'Security',
  'tags': 'transcript, secrets, vault, exposure',
  'text': ' Speaker A: The biggest issue is not just tokens expiring, it is credential exposure through copied config files. Speaker B: Right, we should push teams toward runtime secret injection with the vault agent. Speaker C: And if a leak happens, revoke sessions and rotate the client secret immediately. Speaker A: Please make sure old deployment notes are marked as obsolete because people still find them in shared folders.'},
 {'title': 'SRE Postmortem: Write Path Saturation',
  'doc_id': 'doc22',
  'date': '2026-01-29',
  'team': 'SRE',
  'tags': 'postmortem, write-path, queue, saturation',
  'text': ' The January incident was prolonged because responders initially focused on the gateway symptom instead of the write-path bottleneck. Evidence Store queue depth rose sharply, followed by response deadline breaches and 504 error


## 2. Cleaning and Chunking

### Task
Implement `chunk_documents()` with:
- word-based chunking
- configurable `chunk_size`
- configurable `chunk_overlap`

Each chunk should keep metadata from its parent document.


In [19]:
def clean_text(text: str) -> str:
    text = text.replace("\r", "\n")
    lines = [line.strip() for line in text.splitlines()]
    non_empty = [line for line in lines if line]
    return "\n".join(non_empty)

In [20]:
def chunk_text(text: str, chunk_size: int = 180, chunk_overlap: int = 40) -> list[str]:
    """Split text into overlapping word-based chunks."""
    if chunk_size <= 0:
      raise ValueError("Chunk size must be greater than 0")
    if chunk_size < chunk_overlap:
      raise ValueError("Chunk size must be greater than overlap")
    if chunk_overlap < 0:
      raise ValueError("Overlap must be > 0")

    chunks = []
    for i in range(0, len(text), chunk_size - chunk_overlap):
      chunks.append(text[i : i + chunk_size])

    return chunks

In [21]:
def chunk_documents(documents: list[dict], chunk_size: int = 180, chunk_overlap: int = 40) -> list[Document]:
    """Convert raw document dicts into chunked Document objects.

    Each returned Document must have:
      - text: the chunk string
      - metadata containing at minimum:
          doc_id, title, source_file, chunk_id, date, team, tags

    This is the point where plain dicts become Document objects.
    Everything downstream (retrieval, evaluation, export) expects
    list[Document] - do not return dicts or strings.
    """
    doc_list = []
    for document in documents:
      cleaned_text = clean_text(document["text"])
      chunks = chunk_text(cleaned_text, chunk_size, chunk_overlap)
      for i, text in enumerate(chunks):
        metadata = {key: document[key] for key in document if key != "text"}
        metadata["chunk_id"] = i
        metadata["chunk_size"] = chunk_size
        metadata["chunk_overlap"] = chunk_overlap
        doc_list.append(Document(text, metadata))
    return doc_list


In [22]:
# Inspect one chunk
chunks = chunk_documents(docs, chunk_size=180, chunk_overlap=40)
print(f"Created {len(chunks)} chunks")
print(chunks[0].metadata)
print(chunks[0].text[:250])

Created 89 chunks
{'title': 'Security Review Transcript', 'doc_id': 'doc10', 'date': '2026-02-05', 'team': 'Security', 'tags': 'transcript, secrets, vault, exposure', 'chunk_id': 0, 'chunk_size': 180, 'chunk_overlap': 40}
Speaker A: The biggest issue is not just tokens expiring, it is credential exposure through copied config files. Speaker B: Right, we should push teams toward runtime secret inject



## 3. BM25 retrieval

### Task
Implement:

- a tokenization function (lowercase, alphanumeric terms),
- a `BM25` index built over your chunk corpus,
- a `bm25_retrieve` function that returns chunk-level results with metadata.

In [23]:
def simple_tokenize(text: str) -> list[str]:
    """Lowercase alphanumeric tokenizer."""
    return text.lower().split()

In [24]:
def build_bm25_index(chunks: list[Document]):
    """Build and return a BM25 index plus the tokenized corpus."""
    tokenized_corpus = [simple_tokenize(doc.text) for doc in chunks]
    bm25_index = BM25Okapi(tokenized_corpus)
    return (bm25_index, tokenized_corpus)

In [25]:
def bm25_retrieve(query: str, chunks: list[Document], bm25_index, top_k: int = 5) -> list[Document]:
    """Return top-k chunk results for the query.
    """
    tokenized_query = simple_tokenize(query)
    doc_scores = bm25_index.get_scores(tokenized_query)
    doc_scores_sorted = np.argsort(doc_scores)[::-1]

    chunk_results = []
    for score in doc_scores_sorted[:top_k]:
      chunk_results.append(chunks[score])
    return chunk_results

In [26]:
#Test
bm25_index, bm25_corpus = build_bm25_index(chunks)
test_query = "What does error E4037 mean?"
bm25_results = bm25_retrieve(test_query, chunks, bm25_index, top_k=3)
bm25_results

[Document(text='Error E4037 indicates an expired access token presented to the Gateway API. Error E5042 usually appears when downstream writes stall and requests exceed the response deadline. Erro', metadata={'title': 'Internal Wiki: Error Codes', 'doc_id': 'doc13', 'date': '2026-02-14', 'team': 'Support', 'tags': 'errors, wiki, exact-match', 'chunk_id': 0, 'chunk_size': 180, 'chunk_overlap': 40}),
 Document(text=' malformed input. Resolution: Re-authenticate to obtain a new short-lived access token. Reference: Internal Wiki Error Codes.', metadata={'title': 'Support Case E4037', 'doc_id': 'doc25', 'date': '2026-02-16', 'team': 'Support', 'tags': 'support, error-code, expired-token', 'chunk_id': 1, 'chunk_size': 180, 'chunk_overlap': 40}),
 Document(text='Customer profile exports should be minimized and encrypted. This handbook does not define operational retention for signed execution logs. For Evidence Store retention, consult the', metadata={'title': 'Privacy Handbook', 'doc_id': 'd


## 4. Dense retrieval with sentence embeddings

### Task

Implement:

- dense embedding generation using `sentence-transformers/all-MiniLM-L6-v2`,
- cosine similarity search over chunk embeddings using NumPy or scikit-learn,
- a `dense_retrieve` function.


In [27]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [28]:
def build_dense_index(chunks: list[Document], model) -> np.ndarray:
    """Encode chunk texts and return a 2D numpy array of embeddings."""
    return model.encode([chunk.text for chunk in chunks], normalize_embeddings=True)

In [29]:
def dense_retrieve(query: str, chunks: list[Document], embeddings: np.ndarray, model, top_k: int = 5) -> list[Document]:
    """Dense retrieval using cosine similarity."""
    result = []

    query_embedding = model.encode([query], normalize_embeddings=True)
    doc_embeddings = embeddings
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    sorted = np.argsort(similarities)[::-1]

    for i in sorted[:top_k]:
      result.append(chunks[i])

    return result

In [30]:
dense_embeddings = build_dense_index(chunks, embed_model)
dense_results = dense_retrieve("how do we reduce credential exposure", chunks, dense_embeddings, embed_model, top_k=3)
dense_results

[Document(text='ored in client-side scripts. To reduce credential exposure risk, secrets should be injected at runtime through the vault agent rather than copied into environment files. If a token', metadata={'title': 'API Authentication Policy', 'doc_id': 'doc02', 'date': '2026-02-02', 'team': 'Security', 'tags': 'auth, tokens, credentials, leakage', 'chunk_id': 1, 'chunk_size': 180, 'chunk_overlap': 40}),
 Document(text='Speaker A: The biggest issue is not just tokens expiring, it is credential exposure through copied config files. Speaker B: Right, we should push teams toward runtime secret inject', metadata={'title': 'Security Review Transcript', 'doc_id': 'doc10', 'date': '2026-02-05', 'team': 'Security', 'tags': 'transcript, secrets, vault, exposure', 'chunk_id': 0, 'chunk_size': 180, 'chunk_overlap': 40}),
 Document(text=' push teams toward runtime secret injection with the vault agent. Speaker C: And if a leak happens, revoke sessions and rotate the client secret immediately. S

## 5. Vector database retrieval with ChromaDB
Implement a ChromaDB pipeline. Do not use LangChain or any other abstraction layer in this section.

In [31]:
def get_client():
    CHROMA_DIR.mkdir(parents=True, exist_ok=True)
    return chromadb.PersistentClient(path=str(CHROMA_DIR))

def reset_collection(collection_name: str) -> None:
    client = get_client()
    existing = {c.name for c in client.list_collections()}
    if collection_name in existing:
        client.delete_collection(collection_name)

def get_or_create_collection(collection_name: str):
    client = get_client()
    return client.get_or_create_collection(name=collection_name)

In [32]:
def index_documents_chroma(collection_name: str, chunked_docs: list[Document], model) -> None:
    """Index chunked documents into Chroma using explicit embeddings."""
    # 1. prepare documents, metadatas, ids
    # 2. encode all texts once
    # 3. collection.add(...)

    uids = [str(uuid.uuid4()) for _ in chunked_docs]
    documents = [doc.text for doc in chunked_docs]
    metadatas = [doc.metadata for doc in chunked_docs]

    embeddings = model.encode([doc.text for doc in chunked_docs])
    collection = get_or_create_collection(collection_name)
    collection.add(
        ids = uids,
        documents = documents,
        metadatas = metadatas,
        embeddings = embeddings
    )

In [33]:
def chroma_retrieve(collection_name: str, query: str, top_k: int = 5) -> dict:
    """Run a query against the Chroma collection."""
    collection = get_or_create_collection(collection_name)
    results = collection.query(
      query_texts=[query],
      n_results=top_k
    )
    return results

In [34]:
def chroma_results_to_dataframe(results: dict) -> pd.DataFrame:
    """Convert raw Chroma output into a tidy dataframe for inspection."""
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]

    rows = []
    for rank, (doc, meta, distance) in enumerate(zip(docs, metas, distances), start=1):
        rows.append({
            "rank": rank,
            "doc_id": meta.get("doc_id"),
            "title": meta.get("title"),
            "chunk_id": meta.get("chunk_id"),
            "distance": distance,
            "preview": doc[:220].replace("\n", " "),
        })
    return pd.DataFrame(rows)

In [35]:
# Example indexing workflow
reset_collection(COLLECTION_NAME)
index_documents_chroma(COLLECTION_NAME, chunks, embed_model)
chroma_results = chroma_retrieve(COLLECTION_NAME, "What does error E4037 mean?", top_k=3)
chroma_results_to_dataframe(chroma_results)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 31.7MiB/s]


,rank,doc_id,title,chunk_id,distance,preview
0,1,doc13,Internal Wiki: Error Codes,0,0.608975,Error E4037 indicates an expired access token ...
1,2,doc13,Internal Wiki: Error Codes,1,0.743263,uests exceed the response deadline. Error E220...
2,3,doc25,Support Case E4037,0,0.836110,Ticket summary: Customer could not submit a re...


## 6. Hybrid retrieval with Reciprocal Rank Fusion (RRF)

### Task
Implement a hybrid retriever that merges BM25 and ChromaDB rankings using Reciprocal Rank Fusion (RRF).


In [36]:
def reciprocal_rank_fusion(rank_lists: list[list[str]], rrf_k: int = 60) -> list[str]:
    """Fuse ranked document-id lists using Reciprocal Rank Fusion."""
    scores = {}
    for ranked in rank_lists:
        for rank, item in enumerate(ranked, start=1):
            scores[item] = scores.get(item, 0.0) + 1.0 / (rrf_k + rank)
    return [item for item, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

In [37]:
#bm25
def docs_to_doc_ids(results: list[Document]) -> list[str]:
    return [doc.metadata["doc_id"] for doc in results]
#chroma
def chroma_doc_ids(results: dict) -> list[str]:
    return [meta["doc_id"] for meta in results["metadatas"][0]]

In [38]:
def hybrid_retrieve(query: str, chunks: list[Document], bm25_index, embeddings: np.ndarray, model, collection_name: str, top_k: int = 5, rrf_k: int = 60) -> list[Document]:
    """
    Combine BM25 and ChromaDB rankings using Reciprocal Rank Fusion.
    Returns top-k Document objects so callers can access .text and .metadata.
    rrf_k=60 is the standard default; higher values dampen the influence of top ranks.
    """
    chroma_ranks = chroma_retrieve(collection_name, query, top_k)
    bm25_ranks = bm25_retrieve(query, chunks, bm25_index, top_k)

    bm25_doc_ids = docs_to_doc_ids(bm25_ranks)
    chroma_ids = chroma_doc_ids(chroma_ranks)

    result_doc_ids = reciprocal_rank_fusion([bm25_doc_ids, chroma_ids], rrf_k)
    result_chunks = []

    for doc_id in result_doc_ids[:top_k]:
      for chunk in chunks[::-1]:
        if chunk.metadata["doc_id"] == doc_id:
          result_chunks.append(chunk)
          break

    return result_chunks

In [39]:
results = hybrid_retrieve("What does error E4037 mean?", chunks, bm25_index, dense_embeddings, embed_model, COLLECTION_NAME, top_k=3)
results

[Document(text='quoted verbatim by users, BM25 retrieval is especially useful for support search.', metadata={'title': 'Internal Wiki: Error Codes', 'doc_id': 'doc13', 'date': '2026-02-14', 'team': 'Support', 'tags': 'errors, wiki, exact-match', 'chunk_id': 2, 'chunk_size': 180, 'chunk_overlap': 40}),
 Document(text=' malformed input. Resolution: Re-authenticate to obtain a new short-lived access token. Reference: Internal Wiki Error Codes.', metadata={'title': 'Support Case E4037', 'doc_id': 'doc25', 'date': '2026-02-16', 'team': 'Support', 'tags': 'support, error-code, expired-token', 'chunk_id': 1, 'chunk_size': 180, 'chunk_overlap': 40}),
 Document(text='or Evidence Store retention, consult the architecture note maintained by Platform.', metadata={'title': 'Privacy Handbook', 'doc_id': 'doc18', 'date': '2026-01-18', 'team': 'Security', 'tags': 'privacy, data, retention', 'chunk_id': 1, 'chunk_size': 180, 'chunk_overlap': 40})]


## 7. Build grounded context for answering and prompt design

### Task
- Use a `build_context` helper that merges the top retrieved Document objects into one context string. This context is ready to place into a prompt and helps confirm that your retrieval pipeline is producing readable and traceable results from end to end.
- Then build a prompt for a small local model, or for answer-only evaluation. Your prompt must:
  - rely only on the retrieved context
  - clearly tell the model not to guess or speculate
  - return "Insufficient information in the retrieved documents." if the answer cannot be found in the context

In [40]:
def build_context(results: list[Document], max_chunks: int = 4) -> str:
    """
    Join the top retrieved Document objects into a single context string.
    Each chunk is prefixed with its doc_id and title for traceability.
    """
    selected = results[:max_chunks]
    parts = []
    for doc in selected:
        title = doc.metadata.get("title", "Untitled")
        doc_id = doc.metadata.get("doc_id", "unknown")
        parts.append(f"[{doc_id}] {title}\n{doc.text}")
    return "\n\n".join(parts)

In [41]:

def build_prompt(question: str, context: str) -> str:
    """Build a grounded prompt for a local model."""
    return f"""You are answering questions about the OrbitOps Documents.\n
    Use only the retrieved context below.\n
    Do not use outside knowledge.\n
    If the context does not contain enough information, reply exactly:\n
    Insufficient information in the retrieved documents.\n\n
    Question:\n
    {question}\n\n
    Retrieved context:\n
    {context}\n\n
    Answer:
    """


In [42]:
def generate_answer_from_context(query: str, retrieved_chunks):
    """
    Simple grounded answer generator (no LLM required).
    Returns the most relevant chunk or fallback message.
    """

    if not retrieved_chunks:
        return "Insufficient information in the retrieved documents."

    # simple heuristic: return top chunk if it contains query terms
    top_chunk = retrieved_chunks[0].text.lower()
    query_tokens = set(simple_tokenize(query))

    if any(token in top_chunk for token in query_tokens):
        return retrieved_chunks[0].text.strip()

    return "Insufficient information in the retrieved documents."


In [43]:
q = "What should we do after a suspected token leak?" # possibly change chunking to text chunking
relevant_chunks_for_context = hybrid_retrieve(q, chunks, bm25_index, dense_embeddings, embed_model, COLLECTION_NAME, top_k=4)

print("----- CONTEXT -----")
context = build_context(relevant_chunks_for_context, max_chunks=3)
print(context)

print("\n----- PROMPT -----")
print(build_prompt(q, context)[:1200])


print("\n----- ANSWER -----")
answer = generate_answer_from_context(q, relevant_chunks_for_context)
print(answer)

----- CONTEXT -----
[doc23] Retrieval Failure Examples
tic retrieval can outperform exact keyword matching when the user wording differs from the document wording.

[doc21] FAQ: Account Lockout
E4037, support should first check whether the token expired rather than assuming a lockout.

[doc02] API Authentication Policy
 guidance that suggested storing tokens in shared configuration files is obsolete and must not be followed.

----- PROMPT -----
You are answering questions about the OrbitOps Documents.

    Use only the retrieved context below.

    Do not use outside knowledge.

    If the context does not contain enough information, reply exactly:

    Insufficient information in the retrieved documents.


    Question:

    What should we do after a suspected token leak?


    Retrieved context:

    [doc23] Retrieval Failure Examples
tic retrieval can outperform exact keyword matching when the user wording differs from the document wording.

[doc21] FAQ: Account Lockout
E4037, suppo


## 8. Retrieval evaluation

Evaluate retrieval on `data/public_eval.json`, which is a public evaluation set with gold source documents.

### Task
Implement and report he following metrics using a small evaluation loop `evaluate_retriever`:
- Precision@k
- Recall@k
- Hit@k
- MRR

For a query, count a hit if any returned chunk comes from a gold document ID.

All evaluation metrics must be computed using:
- the same query set
- the same value of k
- the same relevance labels

Do not compare methods using different configurations.

In [44]:

def precision_at_k(retrieved_doc_ids: list[str], gold_doc_ids: list[str], k: int) -> float:
    """Compute Precision@k."""
    relevant = 0.0
    for id in retrieved_doc_ids[:k]:
      if id in gold_doc_ids:
        relevant += 1.0
    return relevant / k


In [45]:

def recall_at_k(retrieved_doc_ids: list[str], gold_doc_ids: list[str], k: int) -> float:
    """Compute Recall@k."""
    relevant = 0.0
    for id in retrieved_doc_ids[:k]:
      if id in gold_doc_ids:
        relevant += 1.0
    if gold_doc_ids:
      return relevant / len(gold_doc_ids)
    return 0


In [46]:

def hit_at_k(retrieved_doc_ids: list[str], gold_doc_ids: list[str], k: int) -> float:
    """Compute Hit@k."""
    for id in retrieved_doc_ids[:k]:
      if id in gold_doc_ids:
        return 1.0
    return 0.0


In [47]:

def reciprocal_rank(retrieved_doc_ids: list[str], gold_doc_ids: list[str]) -> float:
    """Compute Hit@k."""
    for index, id in enumerate(retrieved_doc_ids):
      if id in gold_doc_ids:
        return 1 / (index + 1)
    return 0


In [48]:
def evaluate_retriever(eval_items: list[dict], retrieve_fn, k: int = 3) -> pd.DataFrame:
    rows = []
    for item in eval_items:
        query = item["query"]
        gold = item["gold_docs"]
        predicted = retrieve_fn(query, k)
        rows.append({
            "query": query,
            "precision@k": precision_at_k(predicted, gold, k),
            "recall@k": recall_at_k(predicted, gold, k),
            "hit@k": hit_at_k(predicted, gold, k),
            "rr": reciprocal_rank(predicted, gold),
        })
    return pd.DataFrame(rows)

In [49]:
def summarise_eval(df: pd.DataFrame) -> dict:
    return {
        "precision@k": df["precision@k"].mean(),
        "recall@k": df["recall@k"].mean(),
        "hit@k": df["hit@k"].mean(),
        "mrr": df["rr"].mean(),
    }

In [50]:
# public_eval was loaded in the setup cell above.
bm25_eval_df = evaluate_retriever(public_eval, lambda q, k: docs_to_doc_ids(bm25_retrieve(q, chunks, bm25_index, top_k=k)), k=3)
dense_eval_df = evaluate_retriever(public_eval, lambda q, k: docs_to_doc_ids(dense_retrieve(q, chunks, dense_embeddings, embed_model, top_k=k)), k=3)
chroma_eval_df = evaluate_retriever(public_eval, lambda q, k: chroma_doc_ids(chroma_retrieve(COLLECTION_NAME, q, top_k=k)), k=3)
hybrid_eval_df = evaluate_retriever(public_eval, lambda q, k: docs_to_doc_ids(hybrid_retrieve(q, chunks, bm25_index, dense_embeddings, embed_model, COLLECTION_NAME, top_k=k)), k=3)

summary_table = pd.DataFrame([
    {"method": "bm25", **summarise_eval(bm25_eval_df)},
    {"method": "dense", **summarise_eval(dense_eval_df)},
    {"method": "chroma", **summarise_eval(chroma_eval_df)},
    {"method": "hybrid_rrf", **summarise_eval(hybrid_eval_df)},
])
summary_table

,method,precision@k,recall@k,hit@k,mrr
0,bm25,0.611111,0.847222,1.0,0.916667
1,dense,0.750000,1.027778,1.0,1.000000
2,chroma,0.750000,1.027778,1.0,1.000000
3,hybrid_rrf,0.555556,0.777778,1.0,0.958333



## 9. Faithfulness and hallucination analysis

The `heuristic_faithfulness` is a simple heuristic function that checks whether an answer contains unsupported claims.

### Task
Use at least two concrete examples to discuss whether the retrieved context actually supports a candidate answer. For each example you must:

- show the query,
- show the retrieved context (or a relevant excerpt),
- show the candidate answer,
- state whether the answer is supported or unsupported by the context, and why.

For at least two examples:
- show retrieved context
- show answer
- explicitly label:
  - Supported
  - Unsupported
- explain whether failure is due to:
  - retrieval
  - grounding

In [51]:
STOP_WORDS = {"the","a","an","and","or","to","of","in","for","on","with","is","are","be","by","that","this","it","as","at","from"}

def heuristic_faithfulness(answer: str, context: str) -> dict:
    """Return a small dictionary with a support ratio and a flag.

    Suggested idea:
    - tokenize answer and context
    - ignore common stop words
    - compute fraction of answer tokens seen in context
    """
    answer_terms = {w for w in simple_tokenize(answer) if w not in STOP_WORDS}
    context_terms = set(simple_tokenize(context))
    if not answer_terms:
        return {"support_ratio": 0.0, "flag": "weak"}
    ratio = sum(1 for w in answer_terms if w in context_terms) / len(answer_terms)
    flag = "supported" if ratio >= 0.6 else "weak"
    return {"support_ratio": ratio, "flag": flag}




In [52]:
# Example 1: grounded and relevant answer
query_1 = "How many days of paid sick leave do employees receive each year?"
context_1 = """
Full-time employees receive 10 days of paid sick leave per year.
Unused sick leave does not roll over into the next year.
"""
reference_1 = "Full-time employees receive 10 days of paid sick leave per year."
answer_1 = "Employees receive 10 days of paid sick leave each year."

print("Example 1: grounded answer")
print("Faithfulness     :", heuristic_faithfulness(answer_1, context_1), 3)
print()

# Example 2: partially relevant but hallucinated answer
query_2 = "What is the remote work policy?"
context_2 = """
Employees may work remotely up to two days per week with manager approval.
Remote work is not guaranteed for all roles.
"""
reference_2 = "Employees may work remotely up to two days per week with manager approval."
answer_2 = "Employees may work remotely five days per week and receive a home office allowance."

print("Example 2: hallucinated answer")
print("Faithfulness     :", heuristic_faithfulness(answer_2, context_2), 3)
print()


Example 1: grounded answer
Faithfulness     : {'support_ratio': 0.8888888888888888, 'flag': 'supported'} 3

Example 2: hallucinated answer
Faithfulness     : {'support_ratio': 0.5833333333333334, 'flag': 'weak'} 3



## 10. Export files for marking

This section satisfies the export requirement in criterion 1.6 of the rubric.

In [57]:
import json

summary_table.to_csv("metrics_summary.csv", index=False)

results = {}
for item in public_eval:
    query = item["query"]

    # bm25
    bm25_docs = bm25_retrieve(query, chunks, bm25_index, top_k=3)
    bm25_doc_ids = docs_to_doc_ids(bm25_docs)

    # dense
    dense_docs = dense_retrieve(query, chunks, dense_embeddings, embed_model, top_k=3)
    dense_doc_ids = docs_to_doc_ids(dense_docs)

    # chroma
    chroma_results = chroma_retrieve(COLLECTION_NAME, query, top_k=3)
    chroma_ids = chroma_doc_ids(chroma_results)

    # hybrid_rrf
    rrf_docs = hybrid_retrieve(query, chunks, bm25_index, dense_embeddings, embed_model, COLLECTION_NAME, top_k=3)
    rrf_doc_ids = docs_to_doc_ids(rrf_docs)

    results[query] = {
        "bm25" : bm25_doc_ids,
        "dense" : dense_doc_ids,
        "chroma" : chroma_ids,
        "hybrid_rrf" : rrf_doc_ids
    }

with open("retrieval_results.json", "w") as file:
  json.dump(results, file, indent=2)

In [55]:
from google.colab import files
files.download("retrieval_results.json")
files.download("metrics_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>